In [6]:
import pandas as pd

# 1. Load the dataset
df = pd.read_csv('jochem_data.csv')

# 2. Filter for movies only
df_movies = df[df['Title Type'] == 'Movie'].copy()

# 3. Define the columns to drop
# Note: Matching the exact casing found in your dataset snippet
cols_to_drop = [
    "Const", 
    "Directors", 
    "Num Votes", 
    "Runtime (mins)", 
    "IMDb Rating", 
    "Title Type",
    "URL",
    "Release Date"
]

# 4. Drop the columns
df_cleaned = df_movies.drop(columns=cols_to_drop)

# 5. Show the cleaned result
print(df_cleaned.head())

# Save the cleaned version to a new file
df_cleaned.to_csv('jochem_cleaned_movies.csv', index=False)

   Your Rating  Date Rated                Title       Original Title  Year  \
0            6  2026-02-05            Companion            Companion  2025   
2            8  2026-01-01             La haine             La haine  1995   
3            8  2025-12-29            Dark City            Dark City  1998   
5            6  2025-09-13        A Working Man        A Working Man  2025   
6            7  2025-09-12  The Boondock Saints  The Boondock Saints  1999   

                               Genres  
0                    Thriller, Sci-Fi  
2                        Crime, Drama  
3  Thriller, Mystery, Fantasy, Sci-Fi  
5                    Action, Thriller  
6             Thriller, Action, Crime  


In [16]:
import pandas as pd
import re

# 1. Load Data
friend_df = pd.read_csv('jochem_cleaned_movies.csv')
movie_cols = ['MovieID', 'Title_with_Year', 'Genres_dat']
movies_dat = pd.read_csv('movies.dat', sep='::', names=movie_cols, engine='python', encoding='ISO-8859-1')

# --- CLEANING ---
def clean_title(t):
    return str(t).lower().strip()

friend_df['clean_title'] = friend_df['Title'].apply(clean_title)
movies_dat['Year'] = movies_dat['Title_with_Year'].str.extract(r'\((\d{4})\)')
movies_dat['clean_title'] = movies_dat['Title_with_Year'].str.replace(r'\s\(\d{4}\)', '', regex=True).apply(clean_title)

friend_df['Year'] = friend_df['Year'].astype(str)
movies_dat['Year'] = movies_dat['Year'].astype(str)

# 2. MERGE
merged_df = pd.merge(friend_df, movies_dat, on=['clean_title', 'Year'], how='left')

# 3. FORMATTING
merged_df['Timestamp'] = pd.to_datetime(merged_df['Date Rated']).astype('int64') // 10**9

final_output = merged_df[[
    'MovieID', 
    'Title', 
    'Title_with_Year',
    'Year',
    'Your Rating', 
    'Timestamp'
]]

final_output.columns = ['MovieID', 'Friend_Title', 'Dataset_Title', 'Release_Year', 'Rating', 'Timestamp']

# --- FILTERING BY YEAR (NEW STEP) ---
# Convert to numeric (errors='coerce' handles any weird text, making them NaN)
final_output['Release_Year'] = pd.to_numeric(final_output['Release_Year'], errors='coerce')

# Drop anything after 2021
final_output = final_output[final_output['Release_Year'] <= 2021].copy()

# 4. IDENTIFY MISSING MATCHES
missing_matches = final_output[final_output['MovieID'].isna()]

print(f"Total movies (2021 and older): {len(final_output)}")
print(f"Successfully matched: {final_output['MovieID'].notna().sum()}")
print(f"Missing matches: {len(missing_matches)}")

if len(missing_matches) > 0:
    print("\n--- MOVIES YOU NEED TO FIX MANUALLY (Pre-2022) ---")
    print(missing_matches[['Friend_Title', 'Release_Year', 'Rating']])

# 5. SAVE
final_output.to_csv('friend_ratings_full_list.csv', index=False)

Total movies (2021 and older): 165
Successfully matched: 150
Missing matches: 15

--- MOVIES YOU NEED TO FIX MANUALLY (Pre-2022) ---
                                        Friend_Title  Release_Year  Rating
20                                  Mulholland Drive          2001       6
36                                     28 Days Later          2002       6
49                             The Resistance Banker          2018       7
82                              The Raid: Redemption          2011       8
91   Birdman or (The Unexpected Virtue of Ignorance)          2014       8
99                                       City of God          2002       9
108                                       12 Monkeys          1995       8
123        13 Hours: The Secret Soldiers of Benghazi          2016       8
135                                         Parasite          2019       6
137                 Once Upon a Time... in Hollywood          2019       6
145                           Léon: The Pr

---

In [5]:
import pandas as pd
import surprise # Zie https://surpriselib.com/

In [6]:
ratings = pd.read_csv('https://raw.githubusercontent.com/sidooms/MovieTweetings/master/latest/ratings.dat',
                      delimiter='::', engine='python', header=None,
                      names = ['User_ID', 'Movie_ID', 'Rating', 'Rating_Timestamp'])

In [7]:
ratings

,User_ID,Movie_ID,Rating,Rating_Timestamp
0,1,114508,8,1381006850
1,2,499549,9,1376753198
2,2,1305591,8,1376742507
3,2,1428538,1,1371307089
4,3,75314,1,1595468524
...,...,...,...,...
921393,71705,9893250,10,1613857551
921394,71705,9898858,3,1585958452
921395,71706,172495,10,1587107015
921396,71706,414387,10,1587107852


In [8]:
# Het aantal unieke films
n_unique_movies = ratings['Movie_ID'].nunique()

print(f"Totaal aantal unieke films: {n_unique_movies}")

Totaal aantal unieke films: 38013


In [4]:
ratings.drop(columns=['Rating_Timestamp'], inplace=True)

In [5]:
ratings

,User_ID,Movie_ID,Rating
0,1,114508,8
1,2,499549,9
2,2,1305591,8
3,2,1428538,1
4,3,75314,1
...,...,...,...
921393,71705,9893250,10
921394,71705,9898858,3
921395,71706,172495,10
921396,71706,414387,10


In [6]:
from sklearn.model_selection import train_test_split

train_data, temp_data = train_test_split(ratings, test_size=0.2, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

Train: 737118, Val: 92140, Test: 92140


In [7]:
# Group by Movie_ID and calculate the mean of the Rating column
movie_stats = ratings.groupby('Movie_ID')['Rating'].mean()

# To make it look like a nice table, use reset_index()
movie_stats = movie_stats.reset_index()

# Rename the column for clarity
movie_stats.columns = ['Movie_ID', 'Average_Rating']

print(movie_stats.head())

# Sort by 'mean' (Average_Rating) from highest to lowest
top_movies = movie_stats.sort_values(by='Average_Rating', ascending=False)

print(top_movies.head(10))

   Movie_ID  Average_Rating
0         8             5.0
1        10            10.0
2        12            10.0
3        25             8.0
4        91             6.0
       Movie_ID  Average_Rating
38012  16058736            10.0
2262      50818            10.0
14595    430377            10.0
35584   8098546            10.0
28173   3522198            10.0
32295   5537002            10.0
37762  12788426            10.0
14579    429153            10.0
14568    428470            10.0
32978   5917160            10.0


In [8]:
import pandas as pd

# 1. Calculate the Global Average (C)
# This is the average rating across every single row in your dataset
C = ratings['Rating'].mean()

# 2. Calculate the average (R) and the number of ratings (v) for each movie
movie_stats = ratings.groupby('Movie_ID')['Rating'].agg(['mean', 'count'])

# 3. Determine the minimum threshold (m)
# A common choice is the 70th or 80th percentile of the 'count' column.
# This says: "A movie must have more ratings than 70% of other movies to be trusted."
m = movie_stats['count'].quantile(0.7)

# 4. Define the Bayesian Formula
def bayesian_rating(row, m=m, C=C):
    v = row['count']
    R = row['mean']
    # The formula: (v / (v+m) * R) + (m / (v+m) * C)
    return (v / (v + m) * R) + (m / (v + m) * C)

# 5. Apply the formula to your movie stats
movie_stats['weighted_score'] = movie_stats.apply(bayesian_rating, axis=1)

# 6. Sort and get your Top 3 baseline!
top_3_baseline = movie_stats.sort_values(by='weighted_score', ascending=False).head(3)

print(f"Global Average (C): {C:.2f}")
print(f"Minimum Ratings Threshold (m): {m}")
print("\nTop 3 Movies for your Baseline:")
print(top_3_baseline)

Global Average (C): 7.31
Minimum Ratings Threshold (m): 5.0

Top 3 Movies for your Baseline:
               mean  count  weighted_score
Movie_ID                                  
5512872    9.985836    353        9.948500
4921860   10.000000     48        9.746474
5262972   10.000000     28        9.592822


In [9]:
import pandas as pd

# 1. Calculate the Global Average (C)
# This is the average rating across every single row in your dataset
C = ratings['Rating'].mean()

# 2. Calculate the average (R) and the number of ratings (v) for each movie
movie_stats = ratings.groupby('Movie_ID')['Rating'].agg(['mean', 'count'])

# 3. Determine the minimum threshold (m)
# A common choice is the 70th or 80th percentile of the 'count' column.
# This says: "A movie must have more ratings than 70% of other movies to be trusted."
m = 50

# 4. Define the Bayesian Formula
def bayesian_rating(row, m=m, C=C):
    v = row['count']
    R = row['mean']
    # The formula: (v / (v+m) * R) + (m / (v+m) * C)
    return (v / (v + m) * R) + (m / (v + m) * C)

# 5. Apply the formula to your movie stats
movie_stats['weighted_score'] = movie_stats.apply(bayesian_rating, axis=1)

# 6. Sort and get your Top 3 baseline!
top_3_baseline = movie_stats.sort_values(by='weighted_score', ascending=False).head(3)

print(f"Global Average (C): {C:.2f}")
print(f"Minimum Ratings Threshold (m): {m}")
print("\nTop 3 Movies for your Baseline:")
print(top_3_baseline)

Global Average (C): 7.31
Minimum Ratings Threshold (m): 50

Top 3 Movies for your Baseline:
              mean  count  weighted_score
Movie_ID                                 
5512872   9.985836    353        9.654172
111161    9.388275   1177        9.303693
468569    9.279730    740        9.155230


In [10]:
movie_stats.sort_values(by='count', ascending=False)

,mean,count,weighted_score
Movie_ID,,,
1454468,8.230348,3104,8.215799
816692,8.846154,2964,8.820714
8579674,8.615624,2893,8.593487
993846,8.391626,2842,8.372971
7286456,9.081973,2757,9.050456
...,...,...,...
1124048,3.000000,1,7.228066
1122600,6.000000,1,7.286889
1121986,3.000000,1,7.228066


# 3 meest populair movies aanraden

In [11]:
from sklearn.model_selection import train_test_split
import pandas as pd

train_data, temp_data = train_test_split(ratings, test_size=0.2, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)



movies = pd.read_csv(
    "movies.dat",
    sep="::",
    engine="python",
    names=["Movie_ID", "Title", "Genres"]
)

movie_stats = movie_stats.reset_index()  # needed for merge

movie_stats = train_data.groupby('Movie_ID')['Rating'].agg(['mean', 'count'])
movie_stats_named = movie_stats.merge(movies, on="Movie_ID", how="left")
movie_stats_named.sort_values(by='count', ascending=False)

,Movie_ID,mean,count,Title,Genres
17563,1454468,8.212010,2448,Gravity (2013),Drama|Sci-Fi|Thriller
14653,816692,8.823480,2368,Interstellar (2014),Adventure|Drama|Sci-Fi
32851,8579674,8.625853,2344,1917 (2019),Drama|War
15317,993846,8.412788,2299,The Wolf of Wall Street (2013),Biography|Crime|Drama
31801,7286456,9.068548,2232,Joker (2019),Crime|Drama|Thriller
...,...,...,...,...,...
15680,1074191,7.000000,1,The Betrayed (2008),Crime|Drama|Mystery|Thriller
15679,1073654,7.000000,1,L'allenatore nel pallone 2 (2008),Comedy
15675,1073096,10.000000,1,44 (2007),Drama
15667,1071211,7.000000,1,Elvis & Madona (2010),Comedy|Romance


In [ ]:
# Get the top 3 Movie_IDs based on rating count
top_50_popular_ids = movie_stats.sort_values(by='count', ascending=False).head(50).index.tolist()

print(f"Top 3 Popular Movie IDs: {top_50_popular_ids}")

Top 3 Popular Movie IDs: [1454468, 816692, 8579674]


In [17]:
# Group test data to see what each user actually watched
# Replace 'User_ID' with the actual column name in your ratings file if different
user_val_movies = val_data.groupby('User_ID')['Movie_ID'].apply(set).reset_index()
user_val_movies.columns = ['User_ID', 'actual_movies']

In [18]:
def calculate_precision(actual, predicted):
    # Find intersection
    hits = len(set(predicted) & set(actual))
    return hits / len(predicted)

# Apply calculation to every user
user_val_movies['precision'] = user_val_movies['actual_movies'].apply(
    lambda x: calculate_precision(x, top_3_popular_ids)
)

# Final Average Precision
avg_precision = user_val_movies['precision'].mean()

print(f"Average Precision@3: {avg_precision:.4f}")

Average Precision@3: 0.0125


In [19]:
def calculate_metrics(actual, predicted):
    actual_set = set(actual)
    predicted_set = set(predicted)
    
    # Intersection of predicted and actual
    hits = len(predicted_set & actual_set)
    
    # Precision = hits / total predicted (which is 3 in your case)
    precision = hits / len(predicted_set) if len(predicted_set) > 0 else 0
    
    # Recall = hits / total actual movies the user watched in the test set
    recall = hits / len(actual_set) if len(actual_set) > 0 else 0
    
    # F1 Score
    if (precision + recall) > 0:
        f1 = 2 * (precision * recall) / (precision + recall)
    else:
        f1 = 0
        
    return pd.Series([precision, recall, f1])

# Apply to your dataframe
user_val_movies[['precision', 'recall', 'f1']] = user_val_movies['actual_movies'].apply(
    lambda x: calculate_metrics(x, top_3_popular_ids)
)

# Display the global averages
print(f"Average Precision@3: {user_val_movies['precision'].mean():.4f}")
print(f"Average Recall@3:    {user_val_movies['recall'].mean():.4f}")
print(f"Average F1-Score@3:  {user_val_movies['f1'].mean():.4f}")

Average Precision@3: 0.0125
Average Recall@3:    0.0166
Average F1-Score@3:  0.0116
